In [ ]:
# SIT-N Hack-O-Week: Neural Network Foundations & Model Deployment
# Implementing a 2-layer Neural Network from scratch using NumPy
# Features: Custom Activations, SGD/Adam Optimizers, and Pickle/Joblib Deployment

import numpy as np
import pickle
import joblib

# =====================================================================
# 1. ACTIVATION FUNCTIONS & DERIVATIVES
# =====================================================================

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def sigmoid_derivative(a):
    # a is the output of sigmoid(x)
    return a * (1 - a)

def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)


In [ ]:
def binary_cross_entropy(y_true, y_pred):
    epsilon = 1e-15
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

In [ ]:
class MultiLayerPerceptron:
    def __init__(self, input_dim, hidden_dim, output_dim=1):
        # He initialization for ReLU layer, Xavier for Sigmoid layer
        self.W1 = np.random.randn(input_dim, hidden_dim) * np.sqrt(2.0 / input_dim)
        self.b1 = np.zeros((1, hidden_dim))

        self.W2 = np.random.randn(hidden_dim, output_dim) * np.sqrt(1.0 / hidden_dim)
        self.b2 = np.zeros((1, output_dim))

        # Adam Optimizer parameters
        self.t = 0
        self.m_W1, self.v_W1 = np.zeros_like(self.W1), np.zeros_like(self.W1)
        self.m_b1, self.v_b1 = np.zeros_like(self.b1), np.zeros_like(self.b1)
        self.m_W2, self.v_W2 = np.zeros_like(self.W2), np.zeros_like(self.W2)
        self.m_b2, self.v_b2 = np.zeros_like(self.b2), np.zeros_like(self.b2)

    def forward(self, X):
        # Layer 1 (Hidden) - ReLU activation
        self.Z1 = np.dot(X, self.W1) + self.b1
        self.A1 = relu(self.Z1)

        # Layer 2 (Output) - Sigmoid activation for binary classification
        self.Z2 = np.dot(self.A1, self.W2) + self.b2
        self.A2 = sigmoid(self.Z2)
        return self.A2

    def backward(self, X, y_true):
        m = X.shape[0]

        # Derivative of BCE Loss w.r.t Z2 (Output Pre-activation) simplifies nicely to (A2 - Y)
        dZ2 = self.A2 - y_true

        # Layer 2 gradients
        self.dW2 = np.dot(self.A1.T, dZ2) / m
        self.db2 = np.sum(dZ2, axis=0, keepdims=True) / m

        # Backpropagation into Layer 1
        dA1 = np.dot(dZ2, self.W2.T)
        dZ1 = dA1 * relu_derivative(self.Z1)

        # Layer 1 gradients
        self.dW1 = np.dot(X.T, dZ1) / m
        self.db1 = np.sum(dZ1, axis=0, keepdims=True) / m

    def update_sgd(self, lr):
        # Standard Stochastic Gradient Descent update rule
        self.W1 -= lr * self.dW1
        self.b1 -= lr * self.db1
        self.W2 -= lr * self.dW2
        self.b2 -= lr * self.db2

    def update_adam(self, lr, beta1=0.9, beta2=0.999, epsilon=1e-8):
        self.t += 1

        # Update biased first & second moment estimate for W1, b1
        self.m_W1 = beta1 * self.m_W1 + (1 - beta1) * self.dW1
        self.v_W1 = beta2 * self.v_W1 + (1 - beta2) * (self.dW1 ** 2)
        m_W1_corrected = self.m_W1 / (1 - beta1 ** self.t)
        v_W1_corrected = self.v_W1 / (1 - beta2 ** self.t)
        self.W1 -= lr * m_W1_corrected / (np.sqrt(v_W1_corrected) + epsilon)

        self.m_b1 = beta1 * self.m_b1 + (1 - beta1) * self.db1
        self.v_b1 = beta2 * self.v_b1 + (1 - beta2) * (self.db1 ** 2)
        m_b1_corrected = self.m_b1 / (1 - beta1 ** self.t)
        v_b1_corrected = self.v_b1 / (1 - beta2 ** self.t)
        self.b1 -= lr * m_b1_corrected / (np.sqrt(v_b1_corrected) + epsilon)

        # Update biased first & second moment estimate for W2, b2
        self.m_W2 = beta1 * self.m_W2 + (1 - beta1) * self.dW2
        self.v_W2 = beta2 * self.v_W2 + (1 - beta2) * (self.dW2 ** 2)
        m_W2_corrected = self.m_W2 / (1 - beta1 ** self.t)
        v_W2_corrected = self.v_W2 / (1 - beta2 ** self.t)
        self.W2 -= lr * m_W2_corrected / (np.sqrt(v_W2_corrected) + epsilon)

        self.m_b2 = beta1 * self.m_b2 + (1 - beta1) * self.db2
        self.v_b2 = beta2 * self.v_b2 + (1 - beta2) * (self.db2 ** 2)
        m_b2_corrected = self.m_b2 / (1 - beta1 ** self.t)
        v_b2_corrected = self.v_b2 / (1 - beta2 ** self.t)
        self.b2 -= lr * m_b2_corrected / (np.sqrt(v_b2_corrected) + epsilon)

In [ ]:
if __name__ == "__main__":
    # Toy dataset: XOR gate configurations
    X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
    y = np.array([[0], [1], [1], [0]])

    # Instantiate Model: 2 inputs, 4 hidden units, 1 output unit
    mlp = MultiLayerPerceptron(input_dim=2, hidden_dim=4, output_dim=1)

    print("--- Training MLP on XOR with Adam Optimizer ---")
    epochs = 1000
    learning_rate = 0.05

    for epoch in range(epochs):
        predictions = mlp.forward(X)
        loss = binary_cross_entropy(y, predictions)
        mlp.backward(X, y)
        mlp.update_adam(lr=learning_rate)

        if (epoch + 1) % 200 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:04d} | Loss: {loss:.6f}")

    print("\nFinal Predictions Post-Training:")
    print(mlp.forward(X))

--- Training MLP on XOR with Adam Optimizer ---
Epoch 0001 | Loss: 0.738878
Epoch 0200 | Loss: 0.478588
Epoch 0400 | Loss: 0.477689
Epoch 0600 | Loss: 0.477572
Epoch 0800 | Loss: 0.477457
Epoch 1000 | Loss: 0.477408

Final Predictions Post-Training:
[[6.66637334e-01]
 [6.66637334e-01]
 [6.66637334e-01]
 [8.79127031e-05]]


In [ ]:
 print("\n--- Model Deployment: Saving and Loading Models ---")


--- Model Deployment: Saving and Loading Models ---


In [ ]:
pickle_filename = "mlp_model.pkl"
with open(pickle_filename, "wb") as f:
    pickle.dump(mlp, f)
print(f"[Pickle] Saved model state to '{pickle_filename}'")

with open(pickle_filename, "rb") as f:
    loaded_mlp_pickle = pickle.load(f)
print("[Pickle] Loaded model validation output:\n", loaded_mlp_pickle.forward(X))

[Pickle] Saved model state to 'mlp_model.pkl'
[Pickle] Loaded model validation output:
 [[6.66637334e-01]
 [6.66637334e-01]
 [6.66637334e-01]
 [8.79127031e-05]]


In [ ]:
joblib_filename = "mlp_model.joblib"
joblib.dump(mlp, joblib_filename)
print(f"[Joblib] Saved model state to '{joblib_filename}'")

loaded_mlp_joblib = joblib.load(joblib_filename)
print("[Joblib] Loaded model validation output:\n", loaded_mlp_joblib.forward(X))

[Joblib] Saved model state to 'mlp_model.joblib'
[Joblib] Loaded model validation output:
 [[6.66637334e-01]
 [6.66637334e-01]
 [6.66637334e-01]
 [8.79127031e-05]]
